In [1]:
from opt_targeted_transfers import BinaryGapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = BinaryGapTargetedTransfers(c_bar=2.15, n_regressors=10)

In [4]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset, n_epochs=100)

Fitting conditional binary_gap improvement for transfer size 0.01


100%|██████████| 100/100 [00:01<00:00, 89.96it/s, val loss=0.872]


Fitting conditional binary_gap improvement for transfer size 0.2477777777777778


100%|██████████| 100/100 [00:01<00:00, 92.04it/s, val loss=0.88]


Fitting conditional binary_gap improvement for transfer size 0.4855555555555556


...

100%|██████████| 100/100 [00:00<00:00, 111.55it/s, val loss=0.904]


Fitting conditional binary_gap improvement for transfer size 1.9122222222222223


100%|██████████| 100/100 [00:01<00:00, 84.99it/s, val loss=0.905]


Fitting conditional binary_gap improvement for transfer size 2.15


100%|██████████| 100/100 [00:01<00:00, 91.89it/s, val loss=0.905]


In [5]:
# Precomputation for policy optimization step.
import numpy as np
budgets = np.linspace(0.05, 2.15, 10)
tt.get_opt_transfer_sizes_given_budget_grid(validation_dataset, budgets=budgets)

In [6]:
tt.budget_to_t_map

{0.05: 0.01,
 0.2833333333333333: 0.2477777777777778,
 0.5166666666666667: 0.4855555555555556,
 0.75: 0.7233333333333334,
 0.9833333333333334: 0.9611111111111111,
 1.2166666666666668: 1.4366666666666668,
 1.45: 1.6744444444444444,
 1.6833333333333333: 1.9122222222222223,
 1.9166666666666667: 1.9122222222222223,
 2.15: 1.9122222222222223}

In [7]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=budgets[2])
assignments = tt.run_opt(test_covariate_dataset)

In [8]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.2743708368103455,
 'post_transfer_poverty_rate': 0.47758696416955526,
 'policy_cost_per_capita': 0.4855555555555561,
 'budget': 0.5166666666666667,
 'policy_type': 'binary_gap',
 'd': 2}

In [9]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(budgets[-1])
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 1.1567284008628111e-05,
 'post_transfer_poverty_rate': 0.0003578056650582894,
 'policy_cost_per_capita': 1.912222222222227,
 'budget': 2.15,
 'policy_type': 'binary_gap',
 'd': 2}